In [8]:
import matplotlib.colors as clr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from mcDETECT.utils import *
from scipy import sparse
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

In [9]:
color_dct = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
color_cts = clr.LinearSegmentedColormap.from_list("magma", ["#000003", "#3B0F6F", "#8C2980", "#F66E5B", "#FD9F6C", "#FBFCBF"], N=256)

In [10]:
sample = "Xenium_5K_BC"

# Paths
intermediate_path = f"../../data/{sample}/intermediate_data/"
processed_path = f"../../data/{sample}/processed_data/"
output_path = f"../../output/1_exploration/{sample}/"

In [11]:
# Read data
adata = sc.read_h5ad(intermediate_path + "adata.h5ad")
cell_ids = np.load(processed_path + "cell_ids.npy", allow_pickle=True)
assert adata.shape[0] > cell_ids.shape[0], "cell_ids should be a subset of adata.obs_names"
adata_tumor = adata[adata.obs["cell_id"].isin(cell_ids)].copy()

adata_tumor_cyto = sc.read_h5ad(output_path + "adata_cytoplasmic_embedded.h5ad")
assert adata_tumor.shape[0] == adata_tumor_cyto.shape[0], "adata_tumor and adata_tumor_cyto should have the same number of cells"

In [ ]:
# Clustering (resolution 1.0: in BC this separates interferon from complement and S-phase from G2/M programs; 0.5 merges them)
res_cyto = 1.0
sc.tl.louvain(adata_tumor_cyto, resolution = res_cyto, key_added = "subtype_louvain")
adata_tumor.obs["subtype_louvain_cyto"] = adata_tumor_cyto.obs["subtype_louvain"].values
print(f"Number of clusters (cytoplasmic): {adata_tumor.obs['subtype_louvain_cyto'].nunique()}")

In [ ]:
# Cluster diagnostics: depth and spatial context
# - a cluster whose median cytoplasmic depth differs by > 2x from the others is a depth stratum, not a substate
# - a cluster with median target-gene counts < 20 and no markers is a low-depth residue (flagged below)
# - a cluster close to both immune cells and fibroblasts sits at the stroma border (segmentation spill-over)
from scipy.stats import spearmanr

immune_types = ["CD4+ T cell", "CD8+ T cell", "T cell", "B cell", "Dendritic cell", "Myeloid cell", "Mast cell"]
fibroblast_types = [ct for ct in adata.obs["cell_type_merged"].unique() if "ibro" in str(ct)]
xy_tumor = adata_tumor.obs[["global_x", "global_y"]].values
for name, cell_types in {"dist_immune": immune_types, "dist_fibroblast": fibroblast_types}.items():
    xy_ref = adata.obs.loc[adata.obs["cell_type_merged"].isin(cell_types), ["global_x", "global_y"]].values
    adata_tumor.obs[name] = NearestNeighbors(n_neighbors=1).fit(xy_ref).kneighbors(xy_tumor)[0].ravel()

# cytoplasmic depth (full panel) and raw target-gene counts, recovered from the log-normalized layer
depth = adata_tumor_cyto.obs["compartment_total_counts"].values
lognorm = adata_tumor_cyto.layers["lognorm"]
lognorm = lognorm.toarray() if sparse.issparse(lognorm) else lognorm
adata_tumor.obs["cyto_depth"] = depth
adata_tumor.obs["cyto_target_counts"] = np.expm1(lognorm).sum(axis=1) * depth / 1e4

for k in range(5):
    rho = spearmanr(adata_tumor_cyto.obsm["X_pca"][:, k], np.log10(depth + 1)).correlation
    print(f"Spearman(PC{k + 1}, log10 cytoplasmic depth) = {rho:+.2f}")

groups = adata_tumor.obs.groupby("subtype_louvain_cyto", observed=True)
cluster_stats = pd.DataFrame({
    "n_cells": groups.size(),
    "median_cyto_depth": groups["cyto_depth"].median(),
    "median_cyto_target_counts": groups["cyto_target_counts"].median(),
    "median_cell_area": groups["cell_area"].median(),
    "median_dist_immune": groups["dist_immune"].median(),
    "median_dist_fibroblast": groups["dist_fibroblast"].median(),
})
cluster_stats["low_depth"] = cluster_stats["median_cyto_target_counts"] < 20
cluster_stats = cluster_stats.round(1)
cluster_stats.to_csv(output_path + "cluster_stats_cytoplasmic.csv")
cluster_stats

In [13]:
# UMAP plot
sc.set_figure_params(figsize = (8, 8))
ax = sc.pl.umap(adata_tumor_cyto, color="subtype_louvain", size=10, show=False, title="Cytoplasmic expression")
plt.savefig(output_path + "umap_clustering.png", dpi=300, bbox_inches="tight")
plt.close()

In [14]:
# Scatter plot
sc.set_figure_params(figsize = (8, 6))
ax = sc.pl.scatter(adata_tumor, x = "global_x", y = "global_y", color = "subtype_louvain_cyto", palette = color_dct, size = 2.5, alpha = 1, show = False)
ax.grid(False)
plt.savefig(output_path + "scatter.png", dpi = 300, bbox_inches = "tight")
plt.close()

subtypes = adata_tumor.obs["subtype_louvain_cyto"].astype(str)

for subtype in sorted(subtypes.unique(), key=int):

    is_target = subtypes == subtype

    fig, ax = plt.subplots(figsize=(8, 6))

    # Plot all other tumor cells first
    ax.scatter(adata_tumor.obs.loc[~is_target, "global_x"], adata_tumor.obs.loc[~is_target, "global_y"], c="lightgray", s=0.5, alpha=0.8, linewidths=0)

    # Plot target subtype on top
    ax.scatter(adata_tumor.obs.loc[is_target, "global_x"], adata_tumor.obs.loc[is_target, "global_y"], c=color_dct[int(subtype)], s=1, alpha=0.8, linewidths=0)

    ax.grid(False)
    ax.set_xlabel("global_x")
    ax.set_ylabel("global_y")
    ax.set_title(f"Subtype {subtype}")

    plt.savefig(output_path + f"scatter_{subtype}.png", dpi=300, bbox_inches="tight")
    plt.close()

In [15]:
spatial_programs = {"immune_cell": ["CD4+ T cell", "CD8+ T cell", "T cell", "B cell", "Dendritic cell", "Myeloid cell", "Mast cell"],
                    "tcell": ["CD4+ T cell", "CD8+ T cell", "T cell"]}

subtypes = adata_tumor.obs["subtype_louvain_cyto"].astype(str)

for program_name, cell_types in spatial_programs.items():
    
    adata_immune = adata[adata.obs["cell_type_merged"].isin(cell_types)].copy()

    for subtype in sorted(subtypes.unique(), key=int):

        is_target = subtypes == subtype

        fig, ax = plt.subplots(figsize=(8, 6))

        # Plot all immune cells as the common background
        ax.scatter(adata_immune.obs["global_x"], adata_immune.obs["global_y"], c="lightgray", s=0.5, alpha=0.8, linewidths=0)

        # Plot only the target tumor subtype on top
        ax.scatter(adata_tumor.obs.loc[is_target, "global_x"], adata_tumor.obs.loc[is_target, "global_y"], c=color_dct[int(subtype)], s=1, alpha=0.8, linewidths=0)

        ax.grid(False)
        ax.set_xlabel("global_x")
        ax.set_ylabel("global_y")
        ax.set_title(f"Tumor subtype {subtype}")
        plt.savefig(output_path + f"scatter_{subtype}_{program_name}.png", dpi=300, bbox_inches="tight")
        plt.close()

## DE analysis

In [ ]:
# DE on the log-normalized cytoplasmic expression of the gene set (layers["lognorm"]); X holds the regressed and
# scaled values used for the embedding and is not suitable for DE. With ~1e5 cells almost every gene is significant,
# so markers must also pass an effect-size and an expression-fraction threshold
logfc_thr = 0.5
pct_thr = 0.1

sc.tl.rank_genes_groups(
    adata_tumor_cyto,
    groupby="subtype_louvain",
    method="wilcoxon",
    layer="lognorm",
    use_raw=False,
    pts=True
)

de_dict = {}
for cluster in adata_tumor_cyto.obs["subtype_louvain"].cat.categories:

    df = sc.get.rank_genes_groups_df(adata_tumor_cyto, group=cluster)

    df = df[
        (df["logfoldchanges"] > logfc_thr) &
        (df["pvals_adj"] < 0.05) &
        (df["pct_nz_group"] > pct_thr)
    ]

    df = df.sort_values("scores", ascending=False)
    df.to_csv(output_path + f"DE_cytoplasmic_cluster_{cluster}.csv", index=False)
    de_dict[cluster] = df

    print(f"Cluster {cluster}: {df.shape[0]} marker genes; top: {', '.join(df['names'].head(8))}")

In [ ]:
adata_tumor_cyto.obs["subtype_louvain"].value_counts()

In [ ]:
cluster = "3"

de_dict[cluster].head(30)

In [ ]:
sc.pl.rank_genes_groups(
    adata_tumor_cyto,
    n_genes=20,
    sharey=False
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata_tumor_cyto,
    n_genes=10,
    groupby="subtype_louvain",
    layer="lognorm",
    use_raw=False
)

In [ ]:
sc.pl.pca_loadings(
    adata_tumor_cyto,
    components=[1, 2, 3, 4, 5]
)